# Rogii Wellbore Geology Prediction — v13 anchored seq-CNN inference

Loads a 4-layer 1D conv net (37K params) trained on residuals `TVT - tvt_anchor` and writes `submission.csv`.

- **Anchor**: per-well moving anchor = forward-fill of last observed `TVT_input`.
- **Features**: 17 honest (non-leaky) anchored features — see `FEATS` below.
- **Observed rows**: `TVT_input_missing == 0` rows have predicted residual clipped to 0 (anchor equals truth there).
- **CV**: 5-fold well-grouped mean RMSE = 12.69 ft on the full 404-well train table.

Sources mounted in this notebook:
- Competition data: `/kaggle/input/rogii-wellbore-geology-prediction/`
- Pretrained model + pre-engineered test table: `/kaggle/input/rogii-tvt-anchored-seq-cnn/PyTorch/default/<v>/`

In [ ]:
import os, glob, time, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Discover competition data root (Kaggle mounts may be /kaggle/input/<slug>/
# OR /kaggle/input/competitions/<slug>/ depending on the runner).
comp_candidates = sorted({
    Path(p).parent
    for p in glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
})
assert comp_candidates, 'no sample_submission.csv found under /kaggle/input'
COMP = comp_candidates[0]

# Find the model pickle wherever Kaggle mounted it.
candidates = sorted(glob.glob('/kaggle/input/**/model.pkl', recursive=True))
assert candidates, 'model.pkl not found anywhere under /kaggle/input'
MODEL_DIR = Path(candidates[-1]).parent

print(f'competition data: {COMP}')
print(f'model dir       : {MODEL_DIR}')
print(f'model files     : {[p.name for p in MODEL_DIR.iterdir()]}')
print(f'test typewells  : {len(list((COMP / "test").glob("*__typewell.csv")))}')

In [ ]:
# Load the checkpoint and the pre-engineered test feature table
with open(MODEL_DIR / 'model.pkl', 'rb') as f:
    ckpt = pickle.load(f)
print('checkpoint keys:', list(ckpt.keys()))
print(f'feature_builder: {ckpt["feature_builder"]}')
print(f'CV mean RMSE  : {ckpt["cv_result"]["mean_rmse"]:.3f} ft')
print(f'per-well median: {ckpt["cv_result"]["perwell_median"]:.3f} ft')
print(f'tvt clip range: {ckpt["tvt_clip"]}')

test_df = pd.read_csv(MODEL_DIR / 'test_table_for_regressionLearner.csv', low_memory=False)
print(f'\\ntest table: {len(test_df):,} rows / {test_df["well_id"].nunique()} wells')
test_df = test_df.sort_values(['well_id', 'MD']).reset_index(drop=True)
# placeholders that the feature builder expects but the test table doesn't have
test_df['TVT'] = test_df['TVT_input_imp']  # used as a placeholder; v13 features never read it
for c in ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']:
    if c not in test_df.columns:
        test_df[c] = 0.0
if 'well_class' not in test_df.columns:
    test_df['well_class'] = -1
test_df.head(3)

## Typewell interpolation

Per-well typewell GR(TVT) curves from raw competition CSVs. Used to compute `tw_gr_at_anchor`, `tw_slope_at_anchor`, `delta_gr_lateral`, `inferred_dtvt` — the honest geosteer features that v13 computes at the moving anchor (not at true TVT).

In [ ]:
TYPEWELL_DIR = COMP / 'test'
_typewell_cache = {}

def _load_typewell_curve(well_id):
    if well_id in _typewell_cache:
        return _typewell_cache[well_id]
    path = TYPEWELL_DIR / f'{well_id}__typewell.csv'
    if not path.exists():
        _typewell_cache[well_id] = None
        return None
    tw = pd.read_csv(path, low_memory=False)
    if 'TVT' not in tw.columns or 'GR' not in tw.columns:
        _typewell_cache[well_id] = None
        return None
    tw = tw.dropna(subset=['TVT', 'GR']).sort_values('TVT')
    tvt = tw['TVT'].to_numpy(dtype=np.float64)
    gr  = tw['GR'].to_numpy(dtype=np.float64)
    if len(tvt) < 3:
        _typewell_cache[well_id] = None
        return None
    slope = np.gradient(gr, tvt)
    _typewell_cache[well_id] = (tvt, gr, slope)
    return _typewell_cache[well_id]


def _interp_typewell(well_id, tvt_query):
    curve = _load_typewell_curve(well_id)
    if curve is None:
        return (np.full_like(tvt_query, np.nan, dtype=np.float64),
                np.full_like(tvt_query, np.nan, dtype=np.float64))
    tvt_s, gr_s, slope_s = curve
    in_range = (tvt_query >= tvt_s[0]) & (tvt_query <= tvt_s[-1])
    gr_q = np.interp(tvt_query, tvt_s, gr_s)
    sl_q = np.interp(tvt_query, tvt_s, slope_s)
    gr_q = np.where(in_range, gr_q, np.nan)
    sl_q = np.where(in_range, sl_q, np.nan)
    return gr_q, sl_q

for wid in test_df['well_id'].unique():
    curve = _load_typewell_curve(wid)
    print(f'  typewell {wid}: {"loaded" if curve else "MISSING"}, {0 if curve is None else len(curve[0])} rows')

## Feature builder (anchored_v13)

Reproduces the 17-feature pipeline used at training. Composed of:
1. **Moving anchor** — `tvt_anchor` = ffill of last observed `TVT_input` per well.
2. **Position-from-anchor** — `rows_since_anchor`, `md_since_anchor`, `cum_lateral`.
3. **GR signal** — `GR_imp`, rolling means `GR_roll10/50`, derivatives `gr_d1`, `gr_d2`, `gr_roll10_d1`.
4. **Z signal** — `dZ_dMD`, `z_d1`, `dZ_dMD_d1`, anchor-relative `dz_from_anchor`, `tvt_offset_at_anchor`, `implied_tvt_resid_from_anchor`.
5. **Typewell at anchor** — `tw_gr_at_anchor`, `tw_slope_at_anchor`, `delta_gr_lateral`, `inferred_dtvt`.

In [ ]:
def build_v13_features(df):
    out = df.copy()
    is_obs = out['TVT_input_missing'] == 0
    obs = out['TVT_input_imp'].where(is_obs)

    # 1. Moving anchor
    out['moving_anchor'] = out.groupby('well_id')['TVT_input_imp'].transform(
        lambda s: obs.loc[s.index].ffill())
    end_anchor = out.groupby('well_id')['TVT_input_imp'].transform(
        lambda s: obs.loc[s.index].dropna().iloc[-1] if obs.loc[s.index].notna().any() else np.nan)
    out['moving_anchor'] = out['moving_anchor'].fillna(end_anchor)
    out['tvt_anchor'] = out['moving_anchor']

    # 2. Position since anchor (last observed)
    grp = out.groupby('well_id')
    out['__pos'] = grp.cumcount()
    pos_at_obs = out['__pos'].where(is_obs)
    out['__last_obs_pos'] = grp['__pos'].transform(lambda s: pos_at_obs.loc[s.index].ffill())
    out['rows_since_anchor'] = (out['__pos'] - out['__last_obs_pos']).fillna(out['__pos'])
    md_at_obs = out['MD'].where(is_obs)
    out['__last_obs_md'] = grp['MD'].transform(lambda s: md_at_obs.loc[s.index].ffill())
    out['md_since_anchor'] = (out['MD'] - out['__last_obs_md']).fillna(0.0)

    # 3. GR derivatives
    out['gr_d1'] = grp['GR_imp'].diff().fillna(0.0)
    out['gr_d2'] = grp['gr_d1'].diff().fillna(0.0)
    out['gr_roll10_d1'] = grp['GR_roll10'].diff().fillna(0.0)
    out['z_d1'] = grp['Z'].diff().fillna(0.0)
    out['dZ_dMD_d1'] = grp['dZ_dMD'].diff().fillna(0.0)

    # 4. Z-drift relative to anchor row
    z_at_obs = out['Z'].where(is_obs)
    out['z_at_anchor'] = grp['Z'].transform(lambda s: z_at_obs.loc[s.index].ffill())
    out['z_at_anchor'] = out['z_at_anchor'].fillna(out['Z'])
    out['dz_from_anchor'] = out['Z'] - out['z_at_anchor']
    tvt_at_obs = out['TVT_input_imp'].where(is_obs)
    out['tvt_offset_at_anchor'] = grp['TVT_input_imp'].transform(
        lambda s: tvt_at_obs.loc[s.index].ffill()) - out['z_at_anchor']
    out['tvt_offset_at_anchor'] = out['tvt_offset_at_anchor'].fillna(
        out['TVT_input_imp'] - out['Z'])
    out['implied_tvt_from_z'] = out['Z'] + out['tvt_offset_at_anchor']
    out['implied_tvt_resid_from_anchor'] = out['implied_tvt_from_z'] - out['tvt_anchor']

    # 5. Typewell at the moving anchor
    n = len(out)
    tw_gr = np.full(n, np.nan, dtype=np.float64)
    tw_slope = np.full(n, np.nan, dtype=np.float64)
    anchor_arr = out['tvt_anchor'].to_numpy(dtype=np.float64)
    for wid, idx in out.groupby('well_id').indices.items():
        gr_q, sl_q = _interp_typewell(wid, anchor_arr[idx])
        tw_gr[idx] = gr_q
        tw_slope[idx] = sl_q
    out['tw_gr_at_anchor'] = tw_gr
    out['tw_slope_at_anchor'] = tw_slope
    out['delta_gr_lateral'] = out['GR_imp'].to_numpy() - tw_gr
    safe_slope = np.where(np.abs(tw_slope) < 1e-3, np.nan, tw_slope)
    out['inferred_dtvt'] = np.clip(out['delta_gr_lateral'] / safe_slope, -100.0, 100.0)
    for c in ['tw_gr_at_anchor', 'tw_slope_at_anchor',
              'delta_gr_lateral', 'inferred_dtvt']:
        out[c] = out[c].fillna(0.0)

    return out.drop(columns=[c for c in out.columns if c.startswith('__')])

t0 = time.time()
df_feat = build_v13_features(test_df)
print(f'features built in {time.time()-t0:.1f}s; shape={df_feat.shape}')

FEATS = ckpt['feature_list']
missing = [f for f in FEATS if f not in df_feat.columns]
assert not missing, f'missing features at inference: {missing}'
print(f'all {len(FEATS)} model features present')

## Model architecture and inference

In [ ]:
def build_seq_arch(in_ch):
    return nn.Sequential(
        nn.Conv1d(in_ch, 64, 5, padding=2), nn.ReLU(),
        nn.Conv1d(64, 64, 5, padding=2), nn.ReLU(),
        nn.Conv1d(64, 32, 5, padding=2), nn.ReLU(),
        nn.Conv1d(32, 1, 1),
    )

seq = ckpt['seq_state']
net = build_seq_arch(seq['in_ch'])
state = seq['state_dict']
# checkpoint state_dict may be prefixed with 'net.' (from SeqCNN wrapper) — strip it
if all(k.startswith('net.') for k in state.keys()):
    state = {k[len('net.'):]: v for k, v in state.items()}
net.load_state_dict(state)
net.eval()

def _pick_device():
    """Pick best device that actually works for this small CNN.
    Some Kaggle GPUs throw 'no kernel image is available' on PyTorch builds
    that weren't compiled for that compute capability — verify with a tiny
    forward pass and silently fall back to CPU if it fails."""
    candidates = []
    if torch.cuda.is_available():
        candidates.append(torch.device('cuda'))
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        candidates.append(torch.device('mps'))
    candidates.append(torch.device('cpu'))
    for dev in candidates:
        try:
            probe = build_seq_arch(seq['in_ch']).to(dev)
            with torch.no_grad():
                _ = probe(torch.zeros(1, seq['in_ch'], 16, device=dev))
            return dev
        except Exception as e:
            print(f'  device {dev} unusable: {type(e).__name__}: {str(e)[:120]}')
    return torch.device('cpu')

device = _pick_device()
net.to(device)
print(f'inference device: {device}')
print(f'parameter count : {sum(p.numel() for p in net.parameters()):,}')

In [ ]:
X = df_feat[FEATS].fillna(0).to_numpy(dtype=np.float32)
X = (X - seq['mu']) / seq['sd']

pred_resid = np.zeros(len(df_feat), dtype=np.float64)
t0 = time.time()
with torch.no_grad():
    for wid, idx in df_feat.groupby('well_id').indices.items():
        Xw = torch.from_numpy(X[idx].T.copy()).unsqueeze(0).to(device)  # (1, C, T)
        p = net(Xw).squeeze(1).detach().cpu().numpy().flatten()
        pred_resid[idx] = p
print(f'inference done in {time.time()-t0:.2f}s')

# observed rows have residual exactly 0 by construction
obs_mask = (df_feat['TVT_input_missing'] == 0).to_numpy()
pred_resid = np.where(obs_mask, 0.0, pred_resid)

anchor = df_feat['tvt_anchor'].to_numpy(dtype=np.float64)
pred_tvt = anchor + ckpt.get('shrink', 1.0) * pred_resid
lo, hi = ckpt['tvt_clip']
pred_tvt = np.clip(pred_tvt, lo, hi)

print(f'\\npred_tvt range: {pred_tvt.min():.1f} .. {pred_tvt.max():.1f}')
print(f'pred_tvt per-well summary:')
for wid, idx in df_feat.groupby('well_id').indices.items():
    p = pred_tvt[idx]
    print(f'  {wid}: n={len(p)}, mean={p.mean():.1f}, std={p.std():.2f}, min={p.min():.1f}, max={p.max():.1f}')

## Write submission.csv

Sample submission ids are `<well_id>_<row_idx_in_well>` covering only the lateral portion (idx ≥ first lateral row per well).

In [ ]:
df_feat['_idx_in_well'] = df_feat.groupby('well_id').cumcount()
df_feat['_id'] = df_feat['well_id'].astype(str) + '_' + df_feat['_idx_in_well'].astype(str)

# Three candidate prediction strategies — each one robust to a different failure mode.
# After running, we keep `submission.csv` = the BLEND of the three (median per row),
# which is robust to any single strategy being catastrophically wrong on a given well.

anchor = df_feat['tvt_anchor'].to_numpy(dtype=np.float64)
z_at_anchor = df_feat['z_at_anchor'].to_numpy(dtype=np.float64)
z_now = df_feat['Z'].to_numpy(dtype=np.float64)
lo, hi = ckpt['tvt_clip']

# Strategy A: seq-CNN model on top of anchor (current behaviour)
pred_seq = anchor + ckpt.get('shrink', 1.0) * pred_resid
pred_seq = np.clip(pred_seq, lo, hi)

# Strategy B: physics z-drift. If formation surface is locally flat,
#   TVT(row) = anchor + (Z_anchor − Z_row), so a well that rises (Z up)
#   sees TVT drop. Tune alpha=1.0 = full physics.
PHYSICS_ALPHA = 1.0
pred_phys = anchor + PHYSICS_ALPHA * (z_at_anchor - z_now)
pred_phys = np.clip(pred_phys, lo, hi)

# Strategy C: DTW-style typewell inversion per lateral row.
# For each row, slide a windowed lateral-GR template over the typewell's
# GR(TVT) curve and take the TVT with the smallest windowed SAD.
# Search window: ±300 ft around the anchor (wide enough to catch ~200 ft drift,
# narrow enough that multi-modal matches don't wander far).
def _dtw_invert(well_id, gr_arr, anchor_arr, win=25, band_ft=300.0):
    curve = _load_typewell_curve(well_id)
    if curve is None:
        return np.full_like(gr_arr, np.nan, dtype=np.float64)
    tvt_s, gr_s, _ = curve
    n = len(gr_arr)
    if len(tvt_s) < 2 * win + 3:
        return np.full(n, np.nan, dtype=np.float64)
    from numpy.lib.stride_tricks import sliding_window_view
    tw_windows = sliding_window_view(gr_s, 2 * win + 1)
    tw_centers_tvt = tvt_s[win: len(tvt_s) - win]
    lat = np.pad(gr_arr.astype(np.float64), win, mode='edge')
    lat_windows = sliding_window_view(lat, 2 * win + 1)
    out = np.full(n, np.nan, dtype=np.float64)
    lo_idx = np.searchsorted(tw_centers_tvt, anchor_arr - band_ft, side='left')
    hi_idx = np.searchsorted(tw_centers_tvt, anchor_arr + band_ft, side='right')
    for i in range(n):
        a, b = int(lo_idx[i]), int(hi_idx[i])
        if b - a < 1 or not np.isfinite(anchor_arr[i]):
            continue
        diff = tw_windows[a:b] - lat_windows[i]
        sad = np.abs(diff).sum(axis=1)
        j = int(np.argmin(sad))
        out[i] = tw_centers_tvt[a + j]
    return out

pred_dtw = np.full(len(df_feat), np.nan, dtype=np.float64)
t0 = time.time()
gr_imp_arr = df_feat['GR_imp'].to_numpy(dtype=np.float64)
for wid, idx in df_feat.groupby('well_id').indices.items():
    pred_dtw[idx] = _dtw_invert(wid, gr_imp_arr[idx], anchor[idx], win=25, band_ft=300.0)
# Fall back to anchor where the typewell didn't yield a match.
pred_dtw = np.where(np.isfinite(pred_dtw), pred_dtw, anchor)
pred_dtw = np.clip(pred_dtw, lo, hi)
print(f'DTW inversion done in {time.time()-t0:.1f}s')

# Observed rows: residual is exactly 0 for all three strategies.
obs_mask = (df_feat['TVT_input_missing'] == 0).to_numpy()
pred_seq  = np.where(obs_mask, anchor, pred_seq)
pred_phys = np.where(obs_mask, anchor, pred_phys)
pred_dtw  = np.where(obs_mask, anchor, pred_dtw)

# Per-well summary of each strategy's prediction
for label, p in [('seq (orig)', pred_seq), ('z-physics', pred_phys), ('DTW', pred_dtw)]:
    print(f'\\n{label}:')
    for wid, idx in df_feat.groupby('well_id').indices.items():
        v = p[idx]
        a = anchor[idx]
        print(f'  {wid}: pred range {v.min():.1f}..{v.max():.1f}, mean diff from anchor {np.mean(v-a):+.2f} ft')

In [ ]:
# Write submission.csv = DTW prediction (per user request).
# Also write alt_seq.csv and alt_zphysics.csv as alternates in case we want to
# resubmit with a different strategy.
ss = pd.read_csv(COMP / 'sample_submission.csv')

def _write_sub(predictions, out_path):
    df_feat['_pred'] = predictions
    out = ss.merge(
        df_feat[['_id', '_pred']].rename(columns={'_id': 'id', '_pred': 'tvt'}),
        on='id', how='left', suffixes=('_x', ''))
    if 'tvt_x' in out.columns:
        out = out.drop(columns=['tvt_x'])
    fallback = float(df_feat['tvt_anchor'].median())
    if out['tvt'].isna().any():
        n_miss = out['tvt'].isna().sum()
        out['tvt'] = out['tvt'].fillna(fallback)
        print(f'  {out_path}: {n_miss} rows filled with anchor median {fallback:.2f}')
    out[['id', 'tvt']].to_csv(out_path, index=False)
    return out

print('Writing submissions:')
sub_dtw  = _write_sub(pred_dtw,  'submission.csv')
sub_seq  = _write_sub(pred_seq,  'alt_seq.csv')
sub_phys = _write_sub(pred_phys, 'alt_zphysics.csv')

# Also write the median-blend of all three as a defensive alternate
sub_blend = np.median(np.stack([pred_dtw, pred_seq, pred_phys], axis=0), axis=0)
_write_sub(sub_blend, 'alt_blend.csv')

print('\\nDone. submission.csv = DTW typewell inversion (primary).')
print('Alternates: alt_seq.csv (orig CNN), alt_zphysics.csv (Z-physics), alt_blend.csv (median blend).')

## Notes

- The model adds at most a small per-row correction on top of the moving anchor. Tail wells where the lateral drifts >30 ft from anchor cannot be recovered from the available features (verified by 20+ feature/model variants in the local harness).
- 5-fold well-grouped CV: mean RMSE 12.69 ft, **per-well median 8.65 ft**. The 3-well public LB sample dictates the actual score: a typical sample puts this in the 8–11 ft band.
- To submit: click 'Submit to Competition' in the upper-right of this notebook on Kaggle after the run completes.

# Rogii Wellbore Geology Visualization — Non-Interactive Production Version

This version automatically generates plots for all wells in the test set, using the actual model predictions for the 'projected' section.

In [ ]:
!pip install -q plotly pandas numpy ipywidgets

In [ ]:
import os, glob, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

# 1. Setup Global Predictions Map
df_feat['final_tvt'] = pred_dtw 

# Directories
TEST_DIR = Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction/test')
TRAIN_DIR = Path('/kaggle/input/rogii-wellbore-geology-prediction/train') # Need train for ML

MARKERS = {
    'ANCC': '#6D4C41', 'ASTNU': '#388E3C', 'ASTNL': '#8BC34A', 
    'EGFDU': '#FBC02D', 'EGFDL': '#F57C00', 'BUDA': '#D32F2F'
}
CONSOLIDATED_MARKERS = {'ANCC': '#6D4C41', 'ASTN': '#388E3C', 'EGFD': '#FBC02D', 'BUDA': '#D32F2F'}

def normalize_geology(name):
    if pd.isna(name) or str(name).strip() == '': return 'UNDEFINED'
    g = str(name).strip().upper()
    if g.startswith('ASTN'): return 'ASTN'
    if g.startswith('EGFD'): return 'EGFD'
    return g

# --- Classical ML Strategy for Geology Multi-Classification ---

def prepare_typewell_features(df):
    df = df.copy().sort_values('TVT')
    # Feature 1: Raw GR
    # Feature 2: GR rolling window (smoothed signal)
    df['GR_roll5'] = df['GR'].rolling(5, center=True).mean().fillna(method='ffill').fillna(method='bfill')
    # Feature 3: GR Gradient
    df['GR_diff'] = df['GR'].diff().fillna(0)
    # Feature 4: Normalized Depth within well (Context)
    tvt_min, tvt_max = df['TVT'].min(), df['TVT'].max()
    df['TVT_norm'] = (df['TVT'] - tvt_min) / (tvt_max - tvt_min)
    return df[['GR', 'GR_roll5', 'GR_diff', 'TVT_norm']]

# Global Model Variable
geology_model = None
geology_encoder = None

def train_geology_classifier():
    global geology_model, geology_encoder
    print("Training Classical ML Geology Classifier...")
    
    tw_files = list(TRAIN_DIR.glob('*__typewell.csv'))
    if not tw_files:
        print("Warning: No training typewells found. Using heuristics.")
        return
    
    train_dfs = []
    for f in tw_files:
        df = pd.read_csv(f)
        if 'Geology' in df.columns and 'GR' in df.columns:
            df = df.dropna(subset=['Geology', 'GR', 'TVT'])
            if len(df) > 10:
                features = prepare_typewell_features(df)
                features['label'] = df['Geology']
                train_dfs.append(features)
    
    if not train_dfs:
        print("Warning: No valid training data extracted. Using heuristics.")
        return
        
    full_train = pd.concat(train_dfs)
    X = full_train.drop(columns=['label'])
    y = full_train['label']
    
    from sklearn.preprocessing import LabelEncoder
    geology_encoder = LabelEncoder()
    y_encoded = geology_encoder.fit_transform(y)
    
    geology_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    geology_model.fit(X, y_encoded)
    print(f"Trained on {len(X)} samples across {len(geology_encoder.classes_)} geological classes.")

def predict_geology_ml(t_df):
    global geology_model, geology_encoder
    if 'Geology' in t_df.columns and t_df['Geology'].notna().any():
        return t_df
    
    if geology_model is None:
        train_geology_classifier()
        
    if geology_model is None:
        # Fallback to simple logic if model failed
        t_df['Geology'] = ''
        return t_df
    
    out = t_df.copy()
    X_test = prepare_typewell_features(out)
    
    # Predict
    preds_encoded = geology_model.predict(X_test)
    out['Geology'] = geology_encoder.inverse_transform(preds_encoded)
    
    # Post-process: smooth labels to prevent single-row fluctuations
    out['Geology'] = out['Geology'].rolling(5, center=True).apply(lambda x: pd.Series(x).mode()[0] if not pd.Series(x).mode().empty else x[len(x)//2], raw=False).fillna(method='ffill').fillna(method='bfill')
    
    return out

# --- Visualization Logic ---

def calculate_h_dist(df):
    if len(df) == 0: return pd.Series([], dtype=float)
    x0, y0 = df.iloc[0]['X'], df.iloc[0]['Y']
    return np.sqrt((df['X'] - x0)**2 + (df['Y'] - y0)**2)

def plot_well_production(well_id, max_depth_idx, last_idx):
    h_path = TEST_DIR / f"{well_id}__horizontal_well.csv"
    t_path = TEST_DIR / f"{well_id}__typewell.csv"
    if not h_path.exists(): return
    
    raw_h_df = pd.read_csv(h_path)
    t_df = pd.read_csv(t_path)
    
    # Predict Geology using ML
    t_df = predict_geology_ml(t_df)
    
    pred_subset = df_feat[df_feat['well_id'] == well_id].sort_values('MD')
    obs_df = raw_h_df.iloc[:max_depth_idx+1].copy()
    h_dist = calculate_h_dist(raw_h_df)
    
    eval_zone_actual = raw_h_df.iloc[max_depth_idx+1:last_idx+1]
    rmse_msg = ""
    if 'TVT' in eval_zone_actual.columns and eval_zone_actual['TVT'].notna().any():
        actuals = eval_zone_actual['TVT'].to_numpy()
        preds = pred_subset[pred_subset['_idx_in_well'].isin(eval_zone_actual.index)]['final_tvt'].to_numpy()
        min_len = min(len(actuals), len(preds))
        if min_len > 0:
            error = actuals[:min_len] - preds[:min_len]
            rmse = np.sqrt(np.mean(error**2))
            rmse_msg = f" | Eval RMSE: {rmse:.2f} ft"
    
    fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1, 
                        subplot_titles=(f"Vertical Section - {well_id}{rmse_msg}", f"TVT Correlation - {well_id}"),
                        specs=[[{"secondary_y": False}], [{"secondary_y": True}]])
    
    for m, color in MARKERS.items():
        if m in raw_h_df.columns:
            fig.add_trace(go.Scatter(x=h_dist, y=raw_h_df[m], name=f"Top {m}", 
                                     line=dict(color=color, width=1), legendgroup="geology", showlegend=False), row=1, col=1)
    
    fig.add_trace(go.Scatter(x=h_dist.iloc[:max_depth_idx+1], y=raw_h_df['Z'].iloc[:max_depth_idx+1], 
                             name="Observed Path", line=dict(color='black', width=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=h_dist.iloc[max_depth_idx:], y=raw_h_df['Z'].iloc[max_depth_idx:], 
                             name="Predicted Path", line=dict(color='red', width=2, dash='dot')), row=1, col=1)
    
    fig.add_trace(go.Scatter(x=obs_df.index, y=obs_df['TVT_input'], 
                             name="Observed TVT", line=dict(color='black', width=2)), row=2, col=1)
    
    proj_idx = range(max_depth_idx, min(last_idx + 1, len(raw_h_df)))
    if len(proj_idx) > 0:
        well_preds = pred_subset[pred_subset['_idx_in_well'].isin(proj_idx)]
        fig.add_trace(go.Scatter(x=well_preds['_idx_in_well'], y=well_preds['final_tvt'], 
                                 name="Model TVT (Projected)", line=dict(color='red', width=2, dash='dot')), row=2, col=1)

    fig.add_trace(go.Scatter(x=raw_h_df.index, y=raw_h_df['GR'], name="Gamma Ray", 
                             line=dict(color='#455A64', width=1), opacity=0.3), row=2, col=1, secondary_y=True)

    t_df['norm_geo'] = t_df['Geology'].apply(normalize_geology)
    for g in t_df['norm_geo'].unique():
        sub = t_df[t_df['norm_geo'] == g]
        if len(sub) > 0:
            color = CONSOLIDATED_MARKERS.get(g, '#bdc3c7')
            fig.add_hrect(y0=sub['TVT'].min(), y1=sub['TVT'].max(), fillcolor=color, opacity=0.05, line_width=0, row=2, col=1)

    fig.update_yaxes(title_text="Depth (Z, ft)", row=1, col=1)
    fig.update_yaxes(title_text="TVT (ft)", autorange="reversed", row=2, col=1)
    fig.update_xaxes(title_text="Distance (m)", row=1, col=1)
    fig.update_xaxes(title_text="Row Index", row=2, col=1)
    fig.update_layout(height=800, template="plotly_white", showlegend=True, margin=dict(l=20, r=20, t=60, b=20))
    fig.show()

# --- Execution ---
train_geology_classifier()
print("Generating Production Plots with ML-based Geology Multi-Classification...")
SUB_PATH = Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv')

if SUB_PATH.exists():
    sub_df = pd.read_csv(SUB_PATH)
    for wid in sub_df['id'].str.split('_').str[0].unique():
        well_rows = sub_df[sub_df['id'].str.startswith(wid + '_')]
        eval_indices = well_rows['id'].str.split('_').str[1].astype(int)
        max_depth_idx = eval_indices.min() - 1
        last_idx = eval_indices.max()
        print(f"Plotting Test Well: {wid} (Eval Zone: {max_depth_idx+1} to {last_idx})")
        plot_well_production(wid, max_depth_idx, last_idx)
else:
    test_wells = sorted([f.name.replace('__horizontal_well.csv', '') for f in TEST_DIR.glob('*__horizontal_well.csv')])
    for wid in test_wells:
        h_df = pd.read_csv(TEST_DIR / f"{wid}__horizontal_well.csv")
        last_valid = h_df['TVT_input'].last_valid_index()
        max_idx = len(h_df) - 1
        plot_well_production(wid, last_valid if last_valid is not None else 0, max_idx)
